[Reference](https://medium.com/@GaoDalie_AI/gemma-4-12b-turboquant-mtp-rag-better-ocr-self-hosted-c2cc587bea10$0)

# Query

In [1]:
import os
import re
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

from turbovec import TurboQuantIndex
from langchain_openai import OpenAIEmbeddings
from mlx_vlm import load, generate
import numpy as np
from langchain_ollama import OllamaEmbeddings

embedder = OllamaEmbeddings(model="bge-m3:latest")

# embedder = OpenAIEmbeddings(model="text-embedding-3-small")
model, processor = load("mlx-community/gemma-4-12B-4bit")
draft, _ = load("mlx-community/gemma-4-12B-it-qat-assistant-4bit")

index = TurboQuantIndex.load("index.tq")

query = ""

q_vec = np.array([embedder.embed_query(query)], dtype=np.float32)
_, indices = index.search(q_vec, k=5)

with open("data.txt") as f:
    texts = f.readlines()

retrieved = [texts[i].strip() for i in indices[0]]
retrieved = list(dict.fromkeys(retrieved))

# strip URLs from context
context = "\n".join(re.sub(r'https?://\S+', '', c).strip() for c in retrieved)

print("\n🔍 Retrieved Context:\n")
for i, chunk in enumerate(retrieved, 1):
    print(f"{i}. {chunk}")

messages = [
    {
        "role": "system",
        "content": "You are a strict assistant.\n\nRules:\n1. ONLY use the provided context.\n2. DO NOT use prior knowledge.\n3. If the answer is not fully in the context, say exactly: \"Not found in context\"."
    },
    {
        "role": "user",
        "content": f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"
    }
]

prompt = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    add_special_tokens=False,
)

result = generate(
    model,
    processor,
    prompt=prompt,
    draft_model=draft,
    draft_kind="mtp",
    draft_block_size=3,
    max_tokens=512,
    temperature=0.7,
    verbose=False,
)

answer = result.text if hasattr(result, "text") else str(result)
answer = re.sub(r'<(image|audio|video)\|>', '', answer).strip()

print("\n💡 Answer:\n")
print(answer)

# indexer


In [2]:
from turbovec import TurboQuantIndex
from langchain_ollama import OllamaEmbeddings

embedder = OllamaEmbeddings(model="bge-m3:latest")

with open("data/docs.txt") as f:
    texts = [line.strip() for line in f.readlines() if line.strip()]

vectors = embedder.embed_documents(texts)

index = TurboQuantIndex(dim=len(vectors[0]), bit_width=4)
index.add(vectors)

index.write("index/index.tq")

print("✅ Index built successfully!")

# RAG

In [3]:
import os
import re
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

from turbovec import TurboQuantIndex
from mlx_vlm import load, generate
import numpy as np
import streamlit as st
from langchain_ollama import OllamaEmbeddings


embedder = OllamaEmbeddings(model="bge-m3:latest")

@st.cache_resource
def load_models():
    model, processor = load("mlx-community/gemma-4-12B-4bit")
    draft, _ = load("mlx-community/gemma-4-12B-it-qat-assistant-4bit")
    return model, processor, draft

def load_index(index_path, data_path):
    index = TurboQuantIndex.load(index_path)
    with open(data_path) as f:
        raw = f.read()
    texts = chunk_text(raw)
    return index, texts

def ask(query, index, texts, k=5):
    model, processor, draft = load_models()

    q_vec = np.array([embedder.embed_query(query)], dtype=np.float32)
    _, indices = index.search(q_vec, k=k)
    retrieved = [texts[i] for i in indices[0]]
    context = "\n".join(clean_text(c) for c in retrieved)

    messages = [
        {
            "role": "system",
            "content": "STRICT MODE:\n- Answer ONLY from context\n- No external knowledge\n- If missing: say 'Not found in context'"
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {query}"
        }
    ]

    prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        add_special_tokens=False,
    )

    result = generate(
        model,
        processor,
        prompt=prompt,
        draft_model=draft,
        draft_kind="mtp",
        draft_block_size=3,
        max_tokens=512,
        temperature=0.7,
        verbose=False,
    )

    answer = result.text if hasattr(result, "text") else str(result)
    answer = re.sub(r'<(image|audio|video)\|>', '', answer).strip()

    return answer, retrieved